# 13 · Day 2 종합 캡스톤 — 딥러닝 전처리 파이프라인 (커스텀 커널 + interop)

> **CuPy 2일 집중 코스 — Day 2 / 단원 9 · 마무리 통합 실습 (약 60분)**

Day 2의 핵심을 하나의 **DL 전처리 파이프라인**으로 통합합니다: GPU에서 배치 신호를 표준화→**커스텀 커널** 비선형→
특징 추출(리덕션)→ **무복사로 PyTorch 모델 입력**. 그리고 v0→v1(fuse·메모리)→v2(스트림)로 최적화합니다.

## 무엇을 통합하나
| 단계 | 기법 | 출처 |
|------|------|------|
| 표준화·특징 | ndarray·리덕션 | 02·07 |
| 비선형 커스텀 커널 | `ElementwiseKernel`/`@cupy.fuse` | 07 |
| v1 최적화 | fuse·out=·전송↓ | 05·07 |
| v2 최적화 | 스트림 청크 오버랩 | 06 |
| 모델 입력 | DLPack 무복사 | 12 |

## 목표
- 커스텀 커널을 포함한 전처리를 **end-to-end GPU**로 구성한다.
- v0→v1→v2 단계 최적화 + 정확성 검증 + 무복사 모델 연동.

In [ ]:
import numpy as np, cupy as cp
import matplotlib.pyplot as plt
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare, allclose
print_env()
try:
    import torch; HAS_TORCH = torch.cuda.is_available()
except Exception: HAS_TORCH = False
B, N = 512, 50_000

## Stage 0 — 데이터 생성 (GPU)

(B, N) 잡음 신호 배치. 검증용 host 사본도 둡니다.

In [ ]:

# B: 배치 크기 (데이터 개수), N: 신호 길이
B, N = 1000, 5000 

# 1. 데이터 생성 (GPU)
B, N = 1000, 5000 
# 1. Host (CPU): 컴퓨터 RAM에 더미 노이즈 신호를 생성합니다.
rng = np.random.default_rng(0)
X_np = rng.standard_normal((B, N)).astype(np.float32)

# 2. PCIe 통신: CPU 메모리의 데이터를 GPU 메모리로 복사합니다.
X_cp = cp.asarray(X_np)

print('X_cp shape:', X_cp.shape)

# [시각화] 원본 신호 (첫 번째 샘플)
plt.figure(figsize=(10, 3))
# GPU(CuPy) 데이터를 Matplotlib으로 그리려면 .get()으로 CPU로 가져와야 합니다.
plt.plot(X_cp[0].get(), alpha=0.7, label='Raw Signal')
plt.title('Stage 0: Original Noisy Signal')
plt.legend()
plt.show()

## Stage 1 — 행별 표준화 (TODO)

각 신호(행)를 z-score 표준화 (02, 브로드캐스팅, 장치 비종속).
데이터의 평균을 0, 분산을 1로 맞추는 z-score 표준화 작업을 진행합니다. 

In [ ]:
def standardize(X):
    # TODO: xp 선택 후 (X-행평균)/(행std+1e-6)
    # X가 NumPy 배열이면 np를, CuPy 배열이면 cp를 반환하여 환경에 맞춥니다.
    # xp = 
    # 각 신호(행, axis=1)별로 평균과 표준편차를 계산합니다.
    # mu = 
    # 0으로 나누는 오류를 방지하기 위해 아주 작은 값(1e-6)을 더해줍니다.
    # sd = 
    # 표준화 적용하여 리턴합니다.
    raise NotImplementedError

    xp = cp.get_array_module(X)
    
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6 
    
# 데이터에 표준화 적용
z_cp = standardize(X_cp)

# 📊 [시각화] 표준화된 신호
plt.figure(figsize=(10, 3))
z_sample = z_cp[0].get()
plt.plot(z_sample, color='orange', alpha=0.7, label='Standardized Signal')
plt.title(f'Stage 1: Standardized (Mean: {z_sample.mean():.2f}, Std: {z_sample.std():.2f})')
# z-score이므로 Y축을 -4 ~ 4로 고정하여 변화를 명확히 봅니다.
plt.ylim(-4, 4)
plt.legend()
plt.show()

<details><summary>💡 해답 보기</summary>

```python
def standardize(X):
    xp = cp.get_array_module(X)
    mu = xp.mean(axis=1, keepdims=True); sd = xp.std(axis=1, keepdims=True)+1e-6
    return (xp-mu)/sd
```
</details>

## Stage 2 — 비선형 커스텀 커널 (TODO)

활성화 `g = tanh(z)·exp(-z²)` 를 **커스텀 커널**로(07). 먼저 `ElementwiseKernel`로 작성합니다.

이제 표준화된 데이터에 비선형 활성화 함수 `g = tanh(z)·exp(-z²)` 를 적용합니다. 
CuPy 내장 함수들을 그냥 조합해도 되지만, 커스텀 커널(ElementwiseKernel)을 직접 작성하면 성능을 한계까지 끌어올릴 수 있습니다.

- 비선형 활성화 함수는 왜 필요할까요?
    * 단순한 선형 연산(더하기, 곱하기)만으로는 노이즈 속에서 유의미한 패턴을 걸러내기 어렵습니다. 
    * 비선형 함수를 사용하면 특정 조건의 데이터만 살리고 나머지는 죽이는 식별력을 가질 수 있습니다.
    * 여기서 사용할 함수는 0에 가까운 유효 신호는 강조하고, 극단적으로 크거나 작은 노이즈(이상치)는 0으로 억제하는 역할을 합니다. 

In [ ]:
# TODO: nl = cp.ElementwiseKernel('float32 z','float32 g',
#            'g = tanhf(z) * expf(-z*z);', 'nl_act')
# (또는 @cp.fuse 버전을 v1에서 사용)

# 데이터에 비선형 활성화 적용
g_cp = nl(z_cp)

# 📊 [시각화] 커널의 형태와 활성화된 신호 비교
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# 왼쪽: 커널의 이론적 형태 (어떤 신호를 살리고 죽이는지 확인)
z_range = np.linspace(-5, 5, 200)
axes[0].plot(z_range, np.tanh(z_range) * np.exp(-z_range**2), 'r--')
axes[0].set_title('Kernel Shape: tanh(z) * exp(-z²)')
axes[0].grid(True)

# 오른쪽: 커널을 통과하여 노이즈가 억제된 실제 신호
axes[1].plot(g_cp[0].get(), color='purple', alpha=0.7)
axes[1].set_title('Stage 2: Activated Signal (Noise Suppressed)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

<details><summary>💡 해답 보기</summary>

```python
nl = cp.ElementwiseKernel('float32 z','float32 g',
                          'g = tanhf(z) * expf(-z*z);', 'nl_act')
# 검증용 순수 CuPy 참조
def nl_ref(z): return cp.tanh(z)*cp.exp(-z*z)
```
</details>

## Stage 3 — 특징 추출 & v0 조립

활성화 결과의 **행별 평균**을 특징으로 합니다(간단). 전체 파이프라인을 조립하고 CPU/GPU 검증.

In [ ]:
def features_v0(X):
    z = standardize(X)
    g = nl(z) if cp.get_array_module(X) is cp else (np.tanh(z)*np.exp(-z*z))
    return g.mean(axis=1)        # (B,) 특징

# GPU vs CPU 연산 결과 검증
ref = features_v0(X_np)
out = cp.asnumpy(features_v0(X_cp))
print(f"검증 통과 여부: {np.allclose(ref, out, rtol=1e-3, atol=1e-3)}")

# 📊 [시각화] 최종 추출된 특징(Feature)들의 분포 확인
plt.figure(figsize=(6, 4))
plt.hist(out, bins=30, color='green', alpha=0.7, edgecolor='black')
plt.title('Stage 3: Distribution of Extracted Features (B=1000)')
plt.xlabel('Feature Value')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.show()

## 최적화 v1 — @cupy.fuse + 전송 최소화

비선형을 **융합 커널**로 바꿔 중간배열을 줄이고, 끝까지 GPU에 머무릅니다(05·07).

In [ ]:
# TODO: nl_fused를 cu.fuse를 붙여서 확인해보기.
# def nl_fused(z): 
#  return cp.tanh(z)*cp.exp(-z*z)

def features_v1(X):
    z = standardize(X)
    return nl_fused(z).mean(axis=1)
_ = nl_fused(X_cp[:2])   # 워밍업
allclose(cp.asnumpy(features_v0(X_cp)), cp.asnumpy(features_v1(X_cp)), rtol=1e-3, atol=1e-3, name='v1==v0')


<details><summary>💡 해답 보기</summary>

```python
@cp.fuse()
def nl_fused(z): return cp.tanh(z)*cp.exp(-z*z)
def features_v1(X):
    z = standardize(X)
    return nl_fused(z).mean(axis=1)
_ = nl_fused(X_cp[:2])   # 워밍업
allclose(cp.asnumpy(features_v0(X_cp)), cp.asnumpy(features_v1(X_cp)), rtol=1e-3, atol=1e-3, name='v1==v0')
```
</details>

## 최적화 v2 — 스트림 청크 오버랩

배치를 청크로 나눠 여러 스트림에서 전처리(06). 청크는 독립이라 겹칠 수 있습니다.

In [ ]:
def features_v2(X, nstreams=3, chunk=128):
    out = cp.empty(X.shape[0], dtype=cp.float32)
    streams = [cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i, s0 in enumerate(range(0, X.shape[0], chunk)):
# TODO streams 별로 nl_fused를 호출
      with streams[i % nstreams]:
        out #=

    for st in streams: st.synchronize()
    return out
allclose(cp.asnumpy(features_v1(X_cp)), cp.asnumpy(features_v2(X_cp)), rtol=1e-3, atol=1e-3, name='v2==v1')

<details><summary>💡 해답 보기</summary>

```python
def features_v2(X, nstreams=3, chunk=128):
    out = cp.empty(X.shape[0], dtype=cp.float32)
    streams = [cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i, s0 in enumerate(range(0, X.shape[0], chunk)):
        with streams[i % nstreams]:
            out[s0:s0+chunk] = nl_fused(standardize(X[s0:s0+chunk])).mean(axis=1)
    for st in streams: st.synchronize()
    return out
allclose(cp.asnumpy(features_v1(X_cp)), cp.asnumpy(features_v2(X_cp)), rtol=1e-3, atol=1e-3, name='v2==v1')
```
</details>

## 모델 입력 — DLPack 무복사 (12)

특징을 **복사 없이** PyTorch로 넘겨 간단한 선형층에 통과시킵니다(torch 있을 때).

In [ ]:
feat = features_v1(X_cp).reshape(B, 1)   # (B,1) 특징
if HAS_TORCH:
    t = torch.from_dlpack(feat)          # zero-copy
    W = torch.randn(1, 8, device='cuda')
    y = t @ W                             # 모델 입력으로 사용
    print('model out:', tuple(y.shape))
else:
    print('torch 없음 — 개념: torch.from_dlpack(feat) 로 무복사 입력')

## 성능 비교 & 도전 과제

v0(CPU)→v0/v1/v2(GPU) 시간을 비교하세요.

In [ ]:
# (해답 구현 후 실행)
# print_bench(bench(lambda: features_v0(X_np), n_repeat=3, name='v0 CPU'))
# print_bench(bench(lambda: features_v0(X_cp), n_repeat=5, name='v0 GPU'))
# print_bench(bench(lambda: features_v1(X_cp), n_repeat=5, name='v1 fuse'))
# print_bench(bench(lambda: features_v2(X_cp), n_repeat=5, name='v2 streams'))


### 도전 과제

- 특징을 ReductionKernel/cccl `reduce_into`로 바꿔 보기
- 비선형을 RawKernel(11)로 작성해 fuse와 비교
- pinned+blocking=False로 전송까지 오버랩(06)
- 전체를 CUDA Graph로 캡처(06)


### 도전 과제 1: ReductionKernel로 특징 추출 퓨전(Fusion)
요소별 연산과 평균(Reduce)을 하나로 묶는 기법입니다. 코드를 정의한 후, 이전 단계인 v1 (Elementwise 퓨전)과 성능을 비교합니다.


In [ ]:
import cupy as cp

N_cols = 5000 

# preamble을 사용해 CUDA 디바이스(GPU) 전용 C++ 함수를 미리 정의합니다.
preamble_code = '''
__device__ inline float act_func(float X, float mu, float sd) {
    float z = (X - mu) / sd;
    return tanhf(z) * expf(-z*z);
}
'''

reduce_act = cp.ReductionKernel(
    in_params='float32 X, float32 mu, float32 sd',
    out_params='float32 out',
    map_expr='act_func(X, mu, sd)', # 선언문 없이 깔끔하게 함수만 호출!
    reduce_expr='a + b',
    post_map_expr=f'out = a / {N_cols}.0', 
    identity='0',
    name='reduce_act_kernel',
    preamble=preamble_code # 정의한 함수를 커널에 포함시킵니다.
)

def features_v3_reduction(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6
    return reduce_act(X, mu, sd, axis=1)

# [개별 벤치마크 실행]
print("\n--- [비교] 2단계(v1) vs 3단계(v3 Reduction) ---")
print_bench(bench(lambda: features_v1(X_cp), n_repeat=10, name='v1 (Elementwise)'))
print_bench(bench(lambda: features_v3_reduction(X_cp), n_repeat=10, name='v3 (Reduction)'))

### 도전 과제 2: RawKernel을 이용한 네이티브 CUDA C++ 작성
가장 로우레벨인 CUDA C++ 코드를 직접 작성하는 방식입니다. ReductionKernel이 얼마나 최적화가 잘 되어있는지, Raw 코드와 비교해 봅니다.

In [ ]:
B, N = 1000, 5000 
X_np = np.random.default_rng(0).standard_normal((B, N)).astype(np.float32)
X_cp = cp.asarray(X_np)

# ==========================================
# 1. v3 ReductionKernel 준비
# ==========================================
preamble_code = '''
__device__ inline float act_func(float X, float mu, float sd) {
    float z = (X - mu) / sd;
    return tanhf(z) * expf(-z*z);
}
'''
reduce_act = cp.ReductionKernel(
    in_params='float32 X, float32 mu, float32 sd',
    out_params='float32 out',
    map_expr='act_func(X, mu, sd)',
    reduce_expr='a + b',
    post_map_expr=f'out = a / {N}.0', 
    identity='0',
    name='reduce_act_kernel_final',
    preamble=preamble_code
)

def features_v3_reduction(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6
    return reduce_act(X, mu, sd, axis=1)

# ==========================================
# 2. v4 RawKernel 준비 (사용자 제공 코드)
# ==========================================
raw_code = '''
extern "C" __global__
void raw_act_kernel(const float* X, const float* mu, const float* sd, float* out, int N) {
    int row = blockIdx.x;
    int col = threadIdx.x + blockDim.x * blockIdx.y;
    if (col < N) {
        int idx = row * N + col; 
        float z = (X[idx] - mu[row]) / sd[row];
        out[idx] = tanhf(z) * expf(-z*z);
    }
}
'''
raw_act_kernel = cp.RawKernel(raw_code, 'raw_act_kernel')

def features_v4_raw(X_cp):
    B, N = X_cp.shape
    mu = X_cp.mean(axis=1, keepdims=True)
    sd = X_cp.std(axis=1, keepdims=True) + 1e-6
    out = cp.empty_like(X_cp)
    
    threads_per_block = 256
    blocks_per_row = (N + threads_per_block - 1) // threads_per_block
    
    # 💡 C++의 int(32비트) 시그니처와 맞추기 위해 N을 cp.int32로 명시적 변환
    raw_act_kernel((B, blocks_per_row), (threads_per_block, ), (X_cp, mu, sd, out, cp.int32(N)))
    return out.mean(axis=1)

# ==========================================
# 3. 벤치마크 실행
# ==========================================
print("\n--- [비교] v3 Reduction vs v4 Raw Kernel ---")
print_bench(bench(lambda: features_v3_reduction(X_cp), n_repeat=10, name='v3 (Reduction)'))
print_bench(bench(lambda: features_v4_raw(X_cp), n_repeat=10, name='v4 (Raw Kernel)'))

### 도전 과제 3: Pinned Memory + 비동기 스트림으로 전송 오버랩
데이터 복사 시간(Host to Device)을 연산 시간 뒤로 숨기는 테크닉입니다. 일반적인 동기화 전송 방식과 비교하여 전체(End-to-End) 처리 시간을 확인합니다.

In [ ]:
import cupyx  # 💡 CuPy의 확장 유틸리티 모듈을 추가로 불러옵니다.

B, N = 1000, 5000 

# ==========================================
# [수정된 부분] 안전하게 Pinned Memory 할당하기
# ==========================================
# 포인터를 가져와서 덮어씌우는 복잡한 과정 없이, 처음부터 고정 메모리에 얹혀진 NumPy 배열을 생성합니다.
X_np_pinned = cupyx.empty_pinned((B, N), dtype=np.float32)

# 더미 데이터를 채워 넣습니다.
X_np_pinned[:] = np.random.randn(B, N).astype(np.float32)

# ------------------------------------------
# 이하 기존 코드와 동일하게 진행하시면 됩니다.
# ------------------------------------------
n_chunks = 4
chunk_size = B // n_chunks
streams = [cp.cuda.Stream(non_blocking=True) for _ in range(n_chunks)]
d_X = cp.empty((B, N), dtype=cp.float32) 
results = [None] * n_chunks

def process_overlap():
    for i in range(n_chunks):
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size
        
        with streams[i]:
            d_X[start_idx:end_idx].set(X_np_pinned[start_idx:end_idx], stream=streams[i])
            results[i] = features_v3_reduction(d_X[start_idx:end_idx])
            
    cp.cuda.Stream.null.synchronize()
    return cp.concatenate(results)

def standard_transfer():
    d_X_sync = cp.asarray(X_np_pinned) 
    return features_v3_reduction(d_X_sync)

# 벤치마크 실행 (이전에 정의한 bench, print_bench, features_v3_reduction 함수가 있다고 가정)
print("\n--- [비교] 일반 데이터 전송 vs Pinned 비동기 오버랩 ---")
print_bench(bench(standard_transfer, n_repeat=10, name='순차 전송 + 연산'))
print_bench(bench(process_overlap, n_repeat=10, name='비동기 오버랩 (숨김)'))

### 도전 과제 4: CUDA Graph로 전체 파이프라인 캡처하기
파이썬 환경에서 GPU로 명령을 내릴 때마다 발생하는 오버헤드를 측정합니다. 특히 처리 시간이 짧은 커널을 여러 번 부를 때 Graph의 위력이 드러납니다.

In [ ]:
# ==========================================
# 1. 데이터 및 커널 준비
# ==========================================
B, N = 1000, 5000 
# 💡 cuRAND 충돌 방지를 위해 NumPy로 생성 후 CuPy로 넘깁니다 (가장 안전한 방식)
X_np = np.random.randn(B, N).astype(np.float32)
X_cp = cp.asarray(X_np)

preamble_code = '''
__device__ inline float act_func(float X, float mu, float sd) {
    float z = (X - mu) / sd;
    return tanhf(z) * expf(-z*z);
}
'''
reduce_act = cp.ReductionKernel(
    in_params='float32 X, float32 mu, float32 sd',
    out_params='float32 out',
    map_expr='act_func(X, mu, sd)',
    reduce_expr='a + b',
    post_map_expr=f'out = a / {N}.0', 
    identity='0',
    name='reduce_act_kernel_graph',
    preamble=preamble_code
)

def features_v3_reduction(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-6
    return reduce_act(X, mu, sd, axis=1)

# ==========================================
# 2. CUDA Graph 캡처 및 실행 함수
# ==========================================
# 캡처 구간 밖에서 미리 빈 메모리(Dummy) 할당
X_dummy = cp.empty((B, N), dtype=cp.float32)

stream = cp.cuda.Stream(non_blocking=True)
stream.begin_capture()
out_dummy = features_v3_reduction(X_dummy)
graph = stream.end_capture()

def run_cuda_graph(X_input):
    X_dummy[:] = X_input 
    graph.launch()
    cp.cuda.Stream.null.synchronize()
    return out_dummy.copy()

def run_normal_python(X_input):
    return features_v3_reduction(X_input)

# ==========================================
# 3. 최종 성능 비교
# ==========================================
print("\n--- [비교] 파이썬 커널 호출 vs CUDA Graph 실행 ---")
print_bench(bench(lambda: run_normal_python(X_cp), n_repeat=20, name='일반 커널 호출'))
print_bench(bench(lambda: run_cuda_graph(X_cp), n_repeat=20, name='CUDA Graph 실행'))

## 추가 — 특징 커널화 & 단계 프로파일

**실험 — 단계별 이벤트 프로파일**: 표준화/비선형/특징 각 단계 시간을 재 병목을 찾으세요(06).

In [ ]:
import cupy as cp
evs = [cp.cuda.Event() for _ in range(4)]
evs[0].record()
z = standardize(X_cp);            evs[1].record()
g = nl_fused(z);                  evs[2].record()
feat = g.mean(axis=1);            evs[3].record()
evs[3].synchronize()
for nm, a, b in [('standardize',0,1),('nonlin',1,2),('feature',2,3)]:
    print(f'{nm:>11}: {cp.cuda.get_elapsed_time(evs[a], evs[b]):.3f} ms')

**연습 — 특징을 제곱합으로 커널화**: 특징을 `(g*g).mean(axis=1)`(에너지)로 바꾸고, 비선형+제곱을 **하나의 `@cp.fuse`** 로 융합해 v1과 속도를 비교하세요.

In [ ]:
B, N = 1000, 5000 
X_np = np.random.randn(B, N).astype(np.float32)
X_cp = cp.asarray(X_np)

# TODO: @cp.fuse() def nl_sq(z): return (cp.tanh(z)*cp.exp(-z*z))**2
# def features_energy(X): return nl_sq(standardize(X)).mean(axis=1)
# print("\n--- [비교] C++ ElementwiseKernel vs 파이썬 @cp.fuse ---")
# print_bench(bench(lambda: features_v1(X_cp), n_repeat=20, name='v1 (Elementwise)'))
# print_bench(bench(lambda: features_energy(X_cp), n_repeat=20, name='energy (@cp.fuse)'))

<details><summary>💡 해답 보기</summary>

```python
@cp.fuse()
def nl_sq(z):
    g = cp.tanh(z)*cp.exp(-z*z)
    return g*g
def features_energy(X):
    return nl_sq(standardize(X)).mean(axis=1)
out = features_energy(X_cp)
```
</details>

## 제출물 체크리스트

- [ ] Stage 1~2(표준화·커스텀 커널) 구현 + v0 CPU/GPU 일치
- [ ] v1(fuse)·v2(스트림) 구현 + 일치 검증
- [ ] DLPack 무복사로 PyTorch 입력 연결
- [ ] v0(CPU)→v2(GPU) speedup 표
- [ ] (선택) 도전 과제 1개 + 개선 3줄

**2일 코스 완료!** 수고하셨습니다 — NumPy/SciPy 포팅부터 커스텀 CUDA 커널·프레임워크 통합까지 마쳤습니다.